In [1]:
import pandas as pd
from sklearn.ensemble import IsolationForest

# Dataset
urls = [
    "https://www.google.com",
    "https://www.wikipedia.org",
    "https://www.amazon.com",
    "https://www.microsoft.com",
    "http://example.com",
    "http://login-example.com/verify/12345",
    "http://abc.com/@#$%&123456789",
    "http://very-long-suspicious-example.com/login/verify/account"
]

df = pd.DataFrame({"url": urls})

# Feature extraction
def extract_features(url):
    return [
        len(url),
        url.count("."),
        url.count("-"),
        sum(c.isdigit() for c in url),
        sum(c in "@?=&#$%" for c in url),
        url.count("/"),
        int(url.startswith("https")),
        url.count("@")
    ]

features = df["url"].apply(extract_features)

feature_names = [
    "url_length",
    "dots",
    "hyphens",
    "digits",
    "special_chars",
    "slashes",
    "https",
    "at_symbol"
]

X = pd.DataFrame(features.tolist(), columns=feature_names)

# Isolation Forest
model = IsolationForest(
    contamination=0.25,
    random_state=42
)

df["anomaly"] = model.fit_predict(X)

# Convert result
df["status"] = df["anomaly"].map({
    1: "Normal",
    -1: "Anomalous"
})

print(df[["url", "status"]])

                                                 url     status
0                             https://www.google.com     Normal
1                          https://www.wikipedia.org     Normal
2                             https://www.amazon.com     Normal
3                          https://www.microsoft.com     Normal
4                                 http://example.com     Normal
5              http://login-example.com/verify/12345     Normal
6                      http://abc.com/@#$%&123456789  Anomalous
7  http://very-long-suspicious-example.com/login/...  Anomalous
